# Teste Técnico — Engenheiro de Dados PySpark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, DecimalType, StringType
from pyspark.sql.window import Window
import os

spark = (
    SparkSession.builder
    .appName("teste-tecnico")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [3]:
# Suporta rodar tanto no Docker (/home/jovyan/data) quanto localmente (./data)
DATA_DIR = "/home/jovyan/data" if os.path.exists("/home/jovyan/data") else "data"

clients_schema = StructType([
    StructField("id",   LongType(),   nullable=False),
    StructField("name", StringType(), nullable=True),
])

pedidos_schema = StructType([
    StructField("id",        LongType(),        nullable=False),
    StructField("client_id", LongType(),        nullable=True),
    StructField("value",     DecimalType(5, 2), nullable=True),
])

clients_df = spark.read.schema(clients_schema).json(f"{DATA_DIR}/clients/data.json")
pedidos_df = spark.read.schema(pedidos_schema).json(f"{DATA_DIR}/pedidos/data.json")

print(f"Clients : {clients_df.count():,} registros")
print(f"Pedidos : {pedidos_df.count():,} registros")

Clients : 10,001 registros
Pedidos : 1,100,000 registros


## Data Quality — Exploração e Relatório de Falhas

Antes de implementar as análises, os dados foram inspecionados para identificar problemas de qualidade.
O código abaixo quantifica cada categoria de problema. Em seguida, `falhas_dq` consolida todos os pedidos com falha (`id` + `motivo`).

| Problema | Descrição |
|---|---|
| `value < 0` | Pedidos com valor negativo |
| `client_id` nulo/zero | ID de cliente inválido |
| `client_id` órfão | FK quebrada `client_id` sem correspondência em clients |
| `id` duplicado | Violação de unicidade do identificador do pedido |
| Campos `NULL` | Dados incompletos que inviabilizam o processamento |

**Decisão:** pedidos com problema são reportados em `falhas_dq` e excluídos das análises seguintes.

In [4]:
# --- contagens exploratórias ---
n_negativos  = pedidos_df.filter(F.col("value") < 0).count()
n_id_nulo    = pedidos_df.filter(F.col("id").isNull()).count()
n_value_nulo = pedidos_df.filter(F.col("value").isNull()).count()
n_cid_nulo   = pedidos_df.filter(F.col("client_id").isNull() | (F.col("client_id") == 0)).count()
n_orfaos     = (
    pedidos_df
    .join(clients_df, pedidos_df["client_id"] == clients_df["id"], "left_anti")
    .count()
)
# GROUP BY é mais eficiente que Window aqui: menor IO no shuffle,
# pois só precisamos do id não precisamos carregar todas as colunas.
n_duplicados = (
    pedidos_df.groupBy("id").count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"value < 0            : {n_negativos:,}")
print(f"id nulo              : {n_id_nulo:,}")
print(f"value nulo           : {n_value_nulo:,}")
print(f"client_id nulo/zero  : {n_cid_nulo:,}")
print(f"client_id orfão      : {n_orfaos:,}")
print(f"id duplicado         : {n_duplicados:,}")

value < 0            : 50,000
id nulo              : 0
value nulo           : 50,000
client_id nulo/zero  : 48
client_id orfão      : 0
id duplicado         : 54,491


In [5]:
# --- relatório de falhas ---
neg_df = (
    pedidos_df.filter(F.col("value") < 0)
    .select(F.col("id"), F.lit("valor negativo").alias("motivo"))
)
id_nulo_df = (
    pedidos_df.filter(F.col("id").isNull())
    .select(F.col("id"), F.lit("id nulo").alias("motivo"))
)
val_nulo_df = (
    pedidos_df.filter(F.col("value").isNull())
    .select(F.col("id"), F.lit("valor nulo").alias("motivo"))
)
cid_inv_df = (
    pedidos_df.filter(F.col("client_id").isNull() | (F.col("client_id") == 0))
    .select(F.col("id"), F.lit("client_id inválido").alias("motivo"))
)
orfaos_df = (
    pedidos_df
    .filter(F.col("client_id").isNotNull() & (F.col("client_id") != 0))
    .join(F.broadcast(clients_df), F.col("client_id") == clients_df["id"], "left_anti")
    .select(F.col("id"), F.lit("cliente não encontrado").alias("motivo"))
)
ids_dup = (
    pedidos_df.groupBy("id").count()
    .filter(F.col("count") > 1)
    .select("id")
)
dup_df = (
    pedidos_df.join(ids_dup, on="id", how="inner")
    .select(F.col("id"), F.lit("id duplicado").alias("motivo"))
)

falhas_dq = (
    neg_df.union(id_nulo_df).union(val_nulo_df)
    .union(cid_inv_df).union(orfaos_df).union(dup_df)
    .cache()
)

falhas_dq.show(20, truncate=False)

+-------+--------------+
|id     |motivo        |
+-------+--------------+
|154475 |valor negativo|
|182109 |valor negativo|
|226706 |valor negativo|
|259261 |valor negativo|
|291638 |valor negativo|
|297709 |valor negativo|
|374099 |valor negativo|
|682513 |valor negativo|
|689759 |valor negativo|
|725251 |valor negativo|
|730669 |valor negativo|
|798007 |valor negativo|
|806105 |valor negativo|
|841069 |valor negativo|
|936896 |valor negativo|
|1002221|valor negativo|
|1065095|valor negativo|
|1076607|valor negativo|
|1169878|valor negativo|
|1170790|valor negativo|
+-------+--------------+
only showing top 20 rows



### Plano de Execução — `falhas_dq`

O `.explain("formatted")` abaixo mostra o plano físico otimizado pelo Catalyst. Pontos relevantes:
- **BroadcastHashJoin** no join de órfãos — `clients` (~10 k linhas) é enviado via broadcast, eliminando o shuffle do lado grande
- **HashAggregate (2 fases)** na detecção de duplicatas — agregação parcial em cada executor antes do shuffle reduz tráfego de rede
- **Union** (append) para consolidar as 6 categorias sem shuffle adicional


In [6]:
falhas_dq.explain("formatted")


== Physical Plan ==
AdaptiveSparkPlan (38)
+- InMemoryTableScan (1)
      +- InMemoryRelation (2)
            +- AdaptiveSparkPlan (37)
               +- Union (36)
                  :- Project (5)
                  :  +- Filter (4)
                  :     +- Scan json  (3)
                  :- Project (8)
                  :  +- Filter (7)
                  :     +- Scan json  (6)
                  :- Project (11)
                  :  +- Filter (10)
                  :     +- Scan json  (9)
                  :- Project (14)
                  :  +- Filter (13)
                  :     +- Scan json  (12)
                  :- Project (21)
                  :  +- BroadcastHashJoin LeftAnti BuildRight (20)
                  :     :- Filter (16)
                  :     :  +- Scan json  (15)
                  :     +- BroadcastExchange (19)
                  :        +- Filter (18)
                  :           +- Scan json  (17)
                  +- Project (35)
                     +- SortM

## Agregação por Cliente

Pedidos com falha de qualidade são excluídos via `left_anti` join com `falhas_dq`.  
`clients` (10k linhas) recebe `broadcast` para evitar shuffle do lado maior (1M pedidos).

In [7]:
ids_com_falha = falhas_dq.select("id").filter(F.col("id").isNotNull()).distinct()

pedidos_validos = (
    pedidos_df
    .filter(F.col("id").isNotNull())
    .join(ids_com_falha, on="id", how="left_anti")
)

pedidos_por_cliente = (
    pedidos_validos.alias("p")
    .join(F.broadcast(clients_df.alias("c")), F.col("p.client_id") == F.col("c.id"), "inner")
    .groupBy(F.col("c.id").alias("client_id"), F.col("c.name").alias("name"))
    .agg(
        F.count("*").alias("qtd_pedidos"),
        F.sum(F.col("p.value")).cast(DecimalType(11, 2)).alias("total_value"),
    )
    .orderBy(F.col("total_value").desc())
    .cache()
)

pedidos_por_cliente.show(20, truncate=False)
falhas_dq.unpersist()


+---------+---------------+-----------+-----------+
|client_id|name           |qtd_pedidos|total_value|
+---------+---------------+-----------+-----------+
|123456   |Inês Siqueira  |469734     |23698016.90|
|9047     |Zachary Reis   |61         |4002.08    |
|4494     |Vitor Marques  |68         |3937.06    |
|2695     |Wanda Silva    |61         |3736.53    |
|2756     |Inês Siqueira  |60         |3735.58    |
|8566     |Tereza Leal    |66         |3731.10    |
|6135     |Mariana Melo   |71         |3700.71    |
|5221     |Yasmin Carvalho|65         |3679.35    |
|9266     |Tereza Leal    |67         |3678.17    |
|2379     |Gustavo Pontes |65         |3657.84    |
|8317     |Sofia Castro   |66         |3656.86    |
|7543     |Vitória Andrade|69         |3656.43    |
|849      |Breno Soares   |66         |3651.20    |
|6532     |João Batista   |68         |3650.20    |
|857      |Julio Viana    |62         |3645.72    |
|9147     |Zachary Reis   |59         |3645.27    |
|2781     |I

DataFrame[id: bigint, motivo: string]

### Plano de Execução — `pedidos_por_cliente`

- **SortMergeJoin (left_anti)** para excluir pedidos inválidos via `ids_com_falha` — eficiente para datasets grandes dos dois lados
- **BroadcastHashJoin** para enriquecer com nome do cliente — `clients` (~10 k linhas) é broadcast
- **HashAggregate (2 fases)** para `count(*)` e `sum(value)` — combiner parcial em cada executor antes do shuffle final


In [8]:
pedidos_por_cliente.explain("formatted")


== Physical Plan ==
AdaptiveSparkPlan (60)
+- InMemoryTableScan (1)
      +- InMemoryRelation (2)
            +- AdaptiveSparkPlan (59)
               +- Sort (58)
                  +- Exchange (57)
                     +- HashAggregate (56)
                        +- Exchange (55)
                           +- HashAggregate (54)
                              +- Project (53)
                                 +- BroadcastHashJoin Inner BuildRight (52)
                                    :- Project (48)
                                    :  +- BroadcastHashJoin LeftAnti BuildRight (47)
                                    :     :- Filter (4)
                                    :     :  +- Scan json  (3)
                                    :     +- BroadcastExchange (46)
                                    :        +- HashAggregate (45)
                                    :           +- Exchange (44)
                                    :              +- HashAggregate (43)
                 

## Estatísticas sobre o Valor Total por Cliente

`pedidos_por_cliente` já está em cache — nenhuma recomputação.  
`percentile_approx` usa algoritmo de quantis aproximados (Greenwald-Khanna), eficiente para dados grandes sem coleta completa.

In [12]:
estatisticas = pedidos_por_cliente.select(
    F.mean("total_value").alias("media"),
    F.percentile_approx("total_value", 0.5).alias("mediana"),
    F.percentile_approx("total_value", 0.1).alias("p10"),
    F.percentile_approx("total_value", 0.9).alias("p90"),
)

estatisticas.show(truncate=False)

+-----------+-------+-------+-------+
|media      |mediana|p10    |p90    |
+-----------+-------+-------+-------+
|4746.662696|2368.08|1880.11|2888.44|
+-----------+-------+-------+-------+



## Clientes Acima da Média

Filtra `pedidos_por_cliente` (em cache) onde `total_value > média`. Ordenado por valor crescente.

In [13]:
row = estatisticas.first()
media = row["media"]

acima_da_media = (
    pedidos_por_cliente
    .filter(F.col("total_value") > media)
    .orderBy("total_value")
)

acima_da_media.show(20, truncate=False)

+---------+-------------+-----------+-----------+
|client_id|name         |qtd_pedidos|total_value|
+---------+-------------+-----------+-----------+
|123456   |Inês Siqueira|469734     |23698016.90|
+---------+-------------+-----------+-----------+



## Média Truncada (P10–P90)

Remove outliers das extremidades. Clientes com `total_value` entre o percentil 10 e o percentil 90.  
Reutiliza os escalares `p10`/`p90` extraídos de `estatisticas` (já em cache).

In [8]:
p10 = row["p10"]
p90 = row["p90"]

media_truncada = (
    pedidos_por_cliente
    .filter((F.col("total_value") >= p10) & (F.col("total_value") <= p90))
    .orderBy("total_value")
)

media_truncada.show(20, truncate=False)

+---------+------------------+-----------+-----------+
|client_id|name              |qtd_pedidos|total_value|
+---------+------------------+-----------+-----------+
|2313     |Natália Ribeiro   |40         |1880.11    |
|6807     |Helena Rodrigues  |35         |1880.15    |
|842      |Ulisses Moraes    |39         |1880.32    |
|1906     |Gabriel Alves     |42         |1880.36    |
|476      |Débora Cortez     |41         |1880.37    |
|3233     |Karina Lopes      |38         |1880.39    |
|4114     |Otávio Cardoso    |35         |1880.83    |
|8884     |Letícia Porto     |41         |1880.85    |
|9003     |Diana Lima        |44         |1880.93    |
|2818     |Thiago Araújo     |41         |1881.57    |
|1679     |Gustavo Pontes    |43         |1882.02    |
|8777     |Emerson Figueiredo|38         |1882.03    |
|9325     |Camila Freitas    |40         |1882.05    |
|5302     |Carlos Souza      |41         |1882.15    |
|618      |Thiago Araújo     |44         |1882.18    |
|2838     